# Bet Sizing


Strategy: **sentiment_synchronous**.

## Equal-Weight Bet Sizing

- **Purpose:** Allocate half of each independent book to each direction.
- **Settings:** Five symbols per direction; rank P(meta_action=1); equal pre-cost notional within each direction.
- **Data:** Development predictions available by the entry cutoff.
- **Findings:**
- **Decision:** Leave empty directions and missing entry allocations in cash; do not transfer capital between books.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.preprocessing.market_data import ResearchPaths, sessions
from src.backtesting.portfolio_management import load_prediction_events
from src.backtesting.portfolio_management import entry_schedule, select_synchronous

PROJECT_ROOT = Path.cwd().resolve().parents[2]
from src.backtesting.portfolio_management import PortfolioSettings
PERIOD = "2025-02-01_2025-12-31"
DATA_START = pd.Timestamp("2025-01-01", tz="UTC")
END = pd.Timestamp("2026-01-01", tz="UTC")
HORIZON_BARS = 1000
K = 5
STEP_SIZE = 0.10
INITIAL_AUM = 100_000.0
BROKER_FEE_BPS = 1.0
SLIPPAGE_BPS = 1.0
HALF_LIFE_MINUTES = 390.0
paths = ResearchPaths(PROJECT_ROOT, model_kind="sentiment", period=PERIOD)
STRATEGY = "sentiment_synchronous"
settings = PortfolioSettings(
    k=K, step_size=STEP_SIZE, initial_aum=INITIAL_AUM,
    broker_fee_bps=BROKER_FEE_BPS, slippage_bps=SLIPPAGE_BPS,
    half_life_minutes=HALF_LIFE_MINUTES,
)
events = load_prediction_events(paths, horizon_bars=HORIZON_BARS)
development = events.loc[events.partition.eq("development")]
calendar = sessions(paths, start=DATA_START, end=END)
end = calendar.loc[calendar.close.lt(development.holdout_boundary.iloc[0]), "close"].max()
schedule = entry_schedule(calendar, end)
selected = select_synchronous(development, schedule, settings.k)
selected = selected.loc[selected.exit.notna() & selected.exit.le(end)]
display(selected[["symbol", "event_start", "group", "entry", "primary_side", "meta_probability", "target_weight"]].head(20))

## Capital Allocation

- **Purpose:** Inspect direction budgets without probability-based sizing or price calibration.
- **Settings:** Initial book capital = 50% of the account; direction budget = 50% of book equity.
- **Data:** The first eligible development entry round.
- **Findings:**
- **Decision:** Fixed schedules determine exits; triple-barrier outcomes remain training labels only.

In [ ]:
example = selected.loc[selected.entry.eq(selected.entry.min())].copy()
example["initial_target_notional"] = example.target_weight * settings.initial_aum / 2
display(example[["group", "symbol", "primary_side", "target_weight", "initial_target_notional"]])